# 🛡️ Fake News Detection: Kaggle Training Pipeline
Trained on `clmentbisaillon/fake-and-real-news-dataset` (~45,000 articles).

In [ ]:
import os
import re
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print('Environment initialized successfully!')

## 1. Locate and Load Dataset

In [ ]:
# Find True.csv and Fake.csv in /kaggle/input
true_files = glob.glob('/kaggle/input/**/True.csv', recursive=True)
fake_files = glob.glob('/kaggle/input/**/Fake.csv', recursive=True)

if not true_files or not fake_files:
    # Fallback to local data dir if run locally
    true_files = glob.glob('data/True.csv') or glob.glob('../data/True.csv')
    fake_files = glob.glob('data/Fake.csv') or glob.glob('../data/Fake.csv')

print('Found True CSV:', true_files)
print('Found Fake CSV:', fake_files)

df_true = pd.read_csv(true_files[0])
df_fake = pd.read_csv(fake_files[0])

df_true['label'] = 'REAL'
df_true['label_num'] = 0
df_fake['label'] = 'FAKE'
df_fake['label_num'] = 1

df = pd.concat([df_true, df_fake], axis=0).reset_index(drop=True)
print(f'Combined Dataset: {len(df)} total articles ({len(df_true)} REAL, {len(df_fake)} FAKE)')

## 2. Anti-Leakage Preprocessing
Strip wire agency tags (e.g., `WASHINGTON (Reuters) -`) so the model learns writing style instead of publisher names.

In [ ]:
def clean_anti_leakage(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'^[A-Z\s]{2,25}?\s*\([A-Za-z\s]+\)\s*[-—]\s*', ' ', text)
    text = re.sub(r'\b(reuters|reuters news agency)\b', ' ', text, flags=re.I)
    return text.strip()

df['title'] = df['title'].fillna('').astype(str)
df['text'] = [clean_anti_leakage(t) for t in df['text'].fillna('')]
df['content'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['content'].str.len() > 20].reset_index(drop=True)
print('Cleaned dataset ready:', len(df))

## 3. Feature Extraction (10,000 TF-IDF Features)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
    strip_accents='unicode',
    lowercase=True
)

X = vectorizer.fit_transform(df['content'])
y = df['label_num'].values
print('TF-IDF Matrix Shape:', X.shape)

## 4. Train-Test Split (80/20) & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}')

# Train Linear Support Vector Classifier with Calibration
print('Training Calibrated LinearSVC...')
base_svc = LinearSVC(C=1.0, random_state=42, dual=False)
model = CalibratedClassifierCV(base_svc, cv=3)
model.fit(X_train, y_train)

# Train Logistic Regression as alternative benchmark
print('Training Logistic Regression...')
lr = LogisticRegression(C=2.0, max_iter=1000, solver='liblinear', random_state=42)
lr.fit(X_train, y_train)
print('Training complete!')

## 5. Model Evaluation & Confusion Matrix

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy:  {acc * 100:.2f}%')
print(f'Precision: {prec * 100:.2f}%')
print(f'Recall:    {rec * 100:.2f}%')
print(f'F1 Score:  {f1 * 100:.2f}%')
print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))

# Save Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
labels = np.array([
    [f'TN\n{cm[0,0]}', f'FP (Type I)\n{cm[0,1]}'],
    [f'FN (Type II)\n{cm[1,0]}', f'TP\n{cm[1,1]}']
])
sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
            xticklabels=['REAL (0)', 'FAKE (1)'], yticklabels=['REAL (0)', 'FAKE (1)'])
plt.title(f'Kaggle Cloud Model\nAccuracy: {acc*100:.2f}%', fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.tight_layout()
plt.savefig('confusion_matrix_linear_svc.png', dpi=200)
plt.close()

metrics = {
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1_score': float(f1),
    'confusion_matrix': cm.tolist()
}
with open('evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 6. Export Trained Model & Vectorizer Checkpoints

In [ ]:
joblib.dump(model, 'best_fake_news_model.joblib')
joblib.dump(lr, 'logistic_regression.joblib')
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
print('All model checkpoints successfully exported to working directory!')